### Split AI-results based on diagnostic correctness

In [ ]:
import json
import shutil
from pathlib import Path

PATH = Path(
    "MIRA_EVALUATED_OUTPUTS"
)  

CORRECT_DIR = Path(
    "MIRA_EVALUATED_OUTPUTS_CORRECT"
)
INCORRECT_DIR = Path(
    "MIRA_EVALUATED_OUTPUTS_INCORRECT"
)

In [ ]:
# Remove existing directories to avoid permission issues
if CORRECT_DIR.exists():
    shutil.rmtree(CORRECT_DIR)
if INCORRECT_DIR.exists():
    shutil.rmtree(INCORRECT_DIR)

CORRECT_DIR.mkdir(parents=True, exist_ok=True)
INCORRECT_DIR.mkdir(parents=True, exist_ok=True)

all_files = list(PATH.rglob("*.json"))

CORRECT_HADM_IDS = set()
WRONG_HADM_IDS = set()

for file in all_files:
    with open(file, "r") as f:
        data = json.load(f)
        try:
            is_correct = data["evaluation"]["diagnosis"]["human_match"]
        except KeyError:
            is_correct = data["evaluation"]["diagnosis"]["match"]

        hadm_id = data["metadata"]["hadm_id"]
        if is_correct:
            CORRECT_HADM_IDS.add(hadm_id)
        else:
            WRONG_HADM_IDS.add(hadm_id)

        # Get relative path from PATH to maintain folder structure
        rel_path = file.relative_to(PATH)

        if is_correct:
            dest_path = CORRECT_DIR / rel_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file, dest_path)
        else:
            dest_path = INCORRECT_DIR / rel_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file, dest_path)

In [ ]:
with open("MIRA_correct_hadm_ids.json", "w") as f:
    json.dump(list(CORRECT_HADM_IDS), f)

with open("MIRA_wrong_hadm_ids.json", "w") as f:
    json.dump(list(WRONG_HADM_IDS), f)

#### For the procedures, also split as correct / wrong


In [ ]:
PATH = Path(
    "<<< set your path here >>>"
)


CORRECT_DIR = Path(
    "MIRA_evaluated_procedures_DIAGNOSIS_CORRECT"
)
INCORRECT_DIR = Path(
    "MIRA_evaluated_procedures_DIAGNOSIS_INCORRECT"
)

CORRECT_DIR.mkdir(parents=True, exist_ok=True)
INCORRECT_DIR.mkdir(parents=True, exist_ok=True)


all_files = list(PATH.rglob("*.json"))


for file in all_files:
    with open(file, "r") as f:
        data = json.load(f)
        hadm_id = data["metadata"]["hadm_id"]
        if hadm_id in CORRECT_HADM_IDS:
            is_correct = True
        else:
            is_correct = False

        # Get relative path from PATH to maintain folder structure
        rel_path = file.relative_to(PATH)

        if is_correct:
            dest_path = CORRECT_DIR / rel_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file, dest_path)
        else:
            dest_path = INCORRECT_DIR / rel_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file, dest_path)